# HypatiaX Notebook 01: Data Generation and Dataset Overview

**Paper:** LLMs as Interfaces to Symbolic Discovery: Perfect Extrapolation via Hybrid Architectures  
**Journal:** Journal of Machine Learning Research (2026)  
**Author:** Dr. Ruperto Pedro Bonet Chaple

This notebook documents the synthetic data generation process for the 30 benchmark equations used in the JMLR paper experiments.

---

## 1. Setup and Imports

In [ ]:
from pathlib import Path
import sys

# ── Repo-root resolution ─────────────────────────────────────────────────────
# Works whether the notebook is run from the repo root, the notebooks/ dir,
# or any subdirectory.  Looks for the marker file hypatiax/data/results.
def _find_repo_root() -> Path:
    """Walk up from this notebook until we find hypatiax/data/results."""
    here = Path().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'data' / 'results').exists():
            return candidate
        if (candidate / 'hypatiax' / 'data' / 'results').exists():
            return candidate / 'hypatiax'
    raise FileNotFoundError(
        "Cannot locate repo root.  "
        "Run the notebook from inside the LLM-HypatiaX-PAPERS repository."
    )

REPO_ROOT   = _find_repo_root()
RESULTS_DIR = REPO_ROOT / 'data' / 'results'
FIGURES_DIR = REPO_ROOT / 'notebooks' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Add repo root to sys.path so hypatiax modules are importable
if str(REPO_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT.parent))

print(f"✓ Repo root   : {REPO_ROOT}")
print(f"✓ Results dir : {RESULTS_DIR}")
print(f"✓ Figures dir : {FIGURES_DIR}")

MAIN_FILE    = RESULTS_DIR / 'hybrid_llm_nn/all_domains/hybrid_llm_nn_all_domains_20260115_131438.json'
MAIN_FILE2   = RESULTS_DIR / 'hybrid_llm_nn/all_domains/hybrid_llm_nn_all_domains_20260115_133510.json'

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Plot style
plt.style.use('seaborn-v0_8-paper')
sns.set_palette('husl')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 150})

# Paths

print('✓ Imports loaded')
print(f'✓ Results path: {MAIN_RESULTS}')
print(f'✓ File exists: {MAIN_RESULTS.exists()}')

## 2. Load Experimental Results

In [ ]:
with open(MAIN_RESULTS) as f:
    raw_results = json.load(f)

df = pd.DataFrame(raw_results)

print(f'Total equations: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nDomains: {df["domain"].unique().tolist()}')

## 3. Dataset Overview

In [ ]:
# Domain distribution
domain_counts = df['domain'].value_counts()
print('Equations per domain:')
print(domain_counts.to_string())
print(f'\nTotal: {len(df)} equations')

In [ ]:
# Extract metadata fields
df['equation_name']  = df['metadata'].apply(lambda x: x.get('equation_name', ''))
df['difficulty']     = df['metadata'].apply(lambda x: x.get('difficulty', ''))
df['formula_type']   = df['metadata'].apply(lambda x: x.get('formula_type', ''))
df['ground_truth']   = df['metadata'].apply(lambda x: x.get('ground_truth', ''))
df['protocol']       = df['metadata'].apply(lambda x: x.get('protocol', ''))

# Extract metrics
df['llm_r2']   = df['llm_result'].apply(lambda x: x['metrics']['r2'] if x else None)
df['nn_r2']    = df['nn_result'].apply(lambda x: x['metrics']['r2'] if x else None)
df['eval_r2']  = df['evaluation'].apply(lambda x: x.get('r2', None) if x else None)
df['success']  = df['evaluation'].apply(lambda x: x.get('success', False) if x else False)

print('Extracted fields successfully')
df[['equation_name', 'domain', 'difficulty', 'formula_type', 'decision', 'eval_r2']].head(10)

## 4. Dataset Statistics

In [ ]:
print('=== DATASET SUMMARY ===')
print(f'Total equations:     {len(df)}')
print(f'Domains:             {df["domain"].nunique()}')
print(f'Difficulty levels:   {sorted(df["difficulty"].unique())}')
print(f'Formula types:       {df["formula_type"].nunique()}')
print(f'Protocols:           {df["protocol"].unique().tolist()}')
print()
print('--- Difficulty distribution ---')
print(df['difficulty'].value_counts().to_string())
print()
print('--- Formula type distribution ---')
print(df['formula_type'].value_counts().to_string())
print()
print('--- Decision (LLM vs NN) ---')
print(df['decision'].value_counts().to_string())

## 5. Visualize Dataset Composition

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Equations per domain
ax = axes[0, 0]
domain_counts.plot(kind='bar', ax=ax, color=sns.color_palette('husl', len(domain_counts)), alpha=0.85)
ax.set_title('(a) Equations per Domain')
ax.set_xlabel('Domain')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
for i, v in enumerate(domain_counts):
    ax.text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# (b) Difficulty distribution
ax = axes[0, 1]
diff_counts = df['difficulty'].value_counts()
colors = {'easy': '#2ecc71', 'medium': '#f39c12', 'hard': '#e74c3c'}
bars = ax.bar(diff_counts.index, diff_counts.values,
              color=[colors.get(d, '#3498db') for d in diff_counts.index], alpha=0.85)
ax.set_title('(b) Difficulty Distribution')
ax.set_xlabel('Difficulty')
ax.set_ylabel('Count')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
            str(int(bar.get_height())), ha='center', fontweight='bold')

# (c) Formula types
ax = axes[1, 0]
type_counts = df['formula_type'].value_counts()
type_counts.plot(kind='barh', ax=ax, color='#3498db', alpha=0.85)
ax.set_title('(c) Formula Types')
ax.set_xlabel('Count')

# (d) Decision distribution
ax = axes[1, 1]
decision_counts = df['decision'].value_counts()
ax.pie(decision_counts.values, labels=decision_counts.index,
       autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c', '#3498db'],
       startangle=90)
ax.set_title('(d) Decision: LLM vs NN')

plt.suptitle('HypatiaX Benchmark Dataset Composition', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_dataset_composition.pdf', dpi=300, bbox_inches='tight')
plt.savefig(FIGURES_DIR / '01_dataset_composition.png', dpi=300, bbox_inches='tight')
print('✓ Saved dataset composition figure')
plt.show()

## 6. Full Equation List

In [ ]:
# Display complete equation table
table = df[['domain', 'equation_name', 'difficulty', 'formula_type', 'ground_truth', 'protocol']]
table = table.sort_values(['domain', 'difficulty'])

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 50)
print(f'Complete equation list ({len(table)} equations):')
display(table.reset_index(drop=True))

## 7. Data Generation Process

Each equation's training data was generated synthetically using the following protocol:

- **Protocol A** (mechanics, thermodynamics, electromagnetism, fluid dynamics, optics, quantum): Uniform random sampling over physically meaningful ranges
- **Protocol B** (chemistry, biology, economics, mathematics): Domain-specific sampling respecting physical/mathematical constraints

**Training regime:** 200 samples per equation  
**Test regime:** 50 samples (interpolation)  
**Extrapolation regime:** 50 samples at 10-100× training range

In [ ]:
# Protocol distribution
print('Protocol distribution:')
print(df['protocol'].value_counts().to_string())
print()
print('Protocol A domains:', df[df['protocol']=='A']['domain'].unique().tolist())
print('Protocol B domains:', df[df['protocol']=='B']['domain'].unique().tolist())

---
**Next:** [02_pure_llm_experiments.ipynb](02_pure_llm_experiments.ipynb)